In [ ]:
!pip install -q "transformers==4.35.2" datasets arabert farasapy bitsandbytes


# Finetune AraGPT2-large (Arabic) on our instruction data

We borrow a ready Arabic language model (AraGPT2-large, 792M params) and teach it our `Instruction / Response` style. The base model is downloaded automatically, finetuned for ~1.5h, and saved to `pg/weights_hf/`.

In [ ]:
import os
os.environ['HF_TOKEN'] = 'hf_YOUR_TOKEN_HERE'
try:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient().get_secret('HF_TOKEN')
    if s: os.environ['HF_TOKEN'] = s
except Exception: pass
!git clone https://github.com/BayanDrp/pisto-gpt-64m.git pg


In [ ]:
import base64
print('Overwriting training/finetune_hf.py with embedded REV 8bit-dp-v3 ...')
code = base64.b64decode('IyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyBGaW5ldHVuZSBhIFBSRVRSQUlORUQgQXJhYmljIG1vZGVsIChBcmFHUFQyLWxhcmdlKSBvbiBvdXIKIyBpbnN0cnVjdGlvbiBkYXRhLiBVc2VzIEh1Z2dpbmdGYWNlIHRyYW5zZm9ybWVycyArIFB5VG9yY2guCiMgVGhlIGJvcnJvd2VkIG1vZGVsIGFscmVhZHkgc3BlYWtzIEFyYWJpYzsgd2Ugb25seSBhZGFwdCBpdC4KIyBSZXF1aXJlczogdG9yY2gsIHRyYW5zZm9ybWVycywgZGF0YXNldHMsIGFyYWJlcnQsIGZhcmFzYXB5CiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CmltcG9ydCBzeXMsIG9zLCBtYXRoLCB0aW1lLCBqc29uLCByYW5kb20sIGluc3BlY3QsIHNodXRpbApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBzdWJwcm9jZXNzLCBwa2dfcmVzb3VyY2VzCm9zLmVudmlyb24uc2V0ZGVmYXVsdCgiVE9LRU5JWkVSU19QQVJBTExFTElTTSIsICJmYWxzZSIpCm9zLmVudmlyb24uc2V0ZGVmYXVsdCgiUFlUT1JDSF9DVURBX0FMTE9DX0NPTkYiLCAiZXhwYW5kYWJsZV9zZWdtZW50czpUcnVlIikKCmRlZiBfZW5zdXJlX2NvbXBhdGlibGVfdHJhbnNmb3JtZXJzKCk6CiAgICAjIEFyYUdQVDIncyBjdXN0b20gbW9kZWwgY29kZSBpbXBvcnRzIHRyYW5zZm9ybWVycy5vbm54LCB3aGljaCB3YXMKICAgICMgcmVtb3ZlZCBpbiB0cmFuc2Zvcm1lcnM+PTQuMzYuIEF1dG8tZG93bmdyYWRlIHNvIGl0IGp1c3Qgd29ya3MuCiAgICB0cnk6CiAgICAgICAgaW1wb3J0IHRyYW5zZm9ybWVycwogICAgICAgIHZlciA9IHRyYW5zZm9ybWVycy5fX3ZlcnNpb25fXwogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICB2ZXIgPSAiMCIKICAgIGlmIHZlciA9PSAiMCIgb3IgcGtnX3Jlc291cmNlcy5wYXJzZV92ZXJzaW9uKHZlcikgPj0gcGtnX3Jlc291cmNlcy5wYXJzZV92ZXJzaW9uKCI0LjM2LjAiKToKICAgICAgICBwcmludCgiRG93bmdyYWRpbmcgdHJhbnNmb3JtZXJzIC0+IDQuMzUuMiAoQXJhR1BUMiBuZWVkcyB0cmFuc2Zvcm1lcnMub25ueCkgLi4uIikKICAgICAgICBzdWJwcm9jZXNzLmNoZWNrX2NhbGwoW3N5cy5leGVjdXRhYmxlLCAiLW0iLCAicGlwIiwgImluc3RhbGwiLCAiLXEiLCAidHJhbnNmb3JtZXJzPT00LjM1LjIiXSkKICAgICAgICBvcy5leGVjdihzeXMuZXhlY3V0YWJsZSwgW3N5cy5leGVjdXRhYmxlXSArIHN5cy5hcmd2KQpfZW5zdXJlX2NvbXBhdGlibGVfdHJhbnNmb3JtZXJzKCkKCmltcG9ydCB0b3JjaCBhcyB0aAppbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCmZyb20gdG9yY2gudXRpbHMuZGF0YSBpbXBvcnQgRGF0YXNldCwgRGF0YUxvYWRlcgoKX0hFUkUgICAgICAgPSBQYXRoKF9fZmlsZV9fKS5wYXJlbnQKUk9PVCAgICAgICAgPSBfSEVSRS5wYXJlbnQKQ09ORklHX1BBVEggPSBST09UIC8gImNvbmZpZyIgLyAiZmluZXR1bmVfaGYuanNvbiIKQ09ORklHX0RJUiAgPSBDT05GSUdfUEFUSC5wYXJlbnQKCndpdGggb3BlbihDT05GSUdfUEFUSCwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgIGNmZyA9IGpzb24ubG9hZChmKQoKTU9ERUxfTkFNRSAgID0gY2ZnWyJtb2RlbF9uYW1lIl0KVFJVU1RfUkVNT1RFID0gY2ZnWyJ0cnVzdF9yZW1vdGVfY29kZSJdCk1BWF9MRU4gICAgICA9IGNmZ1sibWF4X2xlbiJdClVTRV9BUkFCRVJUICA9IGNmZ1sidXNlX2FyYWJlcnQiXQp0cmFpbl9jZmcgICAgPSBjZmdbInRyYWluaW5nIl0KZHNfY2ZnICAgICAgID0gY2ZnWyJkYXRhc2V0Il0KU0FWRV9ESVIgICAgID0gKFJPT1QgLyBjZmdbInNhdmVfZGlyIl0pLnJlc29sdmUoKQpTQVZFX0RJUi5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCkhGX1RPS0VOICAgICA9IG9zLmdldGVudigiSEZfVE9LRU4iKSBvciBvcy5nZXRlbnYoIkhVR0dJTkdGQUNFX0hVQl9UT0tFTiIpCgpkZXZpY2UgPSB0aC5kZXZpY2UoImN1ZGEiIGlmIHRoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikKcHJpbnQoZiJEZXZpY2UgOiB7ZGV2aWNlfSIpCmlmIHRoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICBwcmludChmIkdQVSAgICA6IHt0aC5jdWRhLmdldF9kZXZpY2VfbmFtZSgwKX0iKQpwcmludCgiZmluZXR1bmVfaGYucHkgUkVWOiA4Yml0LWRwLXYzIChiaXRzYW5kYnl0ZXMgOC1iaXQgQWRhbSArIGN1c3RvbS1jb2RlIGNvcHkpIikKCiMg4pSA4pSAIEFyYWJpYyBwcmVwcm9jZXNzaW5nIChSRVFVSVJFRCBmb3IgQXJhR1BUMikg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACnByZXAgPSBOb25lCmlmIFVTRV9BUkFCRVJUOgogICAgdHJ5OgogICAgICAgIGZyb20gYXJhYmVydC5wcmVwcm9jZXNzIGltcG9ydCBBcmFiZXJ0UHJlcHJvY2Vzc29yCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIGZyb20gYXJhYmVydC5hcmFiZXJ0X3ByZXByb2Nlc3NvciBpbXBvcnQgQXJhYmVydFByZXByb2Nlc3NvcgogICAgdHJ5OgogICAgICAgIHByZXAgPSBBcmFiZXJ0UHJlcHJvY2Vzc29yKG1vZGVsX25hbWU9TU9ERUxfTkFNRSkKICAgICAgICBwcmludCgiQXJhYmVydFByZXByb2Nlc3NvciByZWFkeSDinJMiKQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgIHByaW50KGYi4pqgIEFyYWJlcnRQcmVwcm9jZXNzb3IgdW5hdmFpbGFibGUgKHtlfSk7IGNvbnRpbnVpbmcgV0lUSE9VVCBBcmFiaWMgcHJlcHJvY2Vzc2luZyIpCgpkZWYgY2xlYW4odGV4dCk6CiAgICB0ZXh0ID0gc3RyKHRleHQpLnN0cmlwKCkKICAgIGlmIHByZXAgaXMgbm90IE5vbmU6CiAgICAgICAgdHJ5OgogICAgICAgICAgICB0ZXh0ID0gcHJlcC5wcmVwcm9jZXNzKHRleHQpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgcGFzcwogICAgcmV0dXJuIHRleHQKCiMg4pSA4pSAIExvYWQgcHJldHJhaW5lZCBtb2RlbCArIHRva2VuaXplciDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IEF1dG9Nb2RlbEZvckNhdXNhbExNLCBHUFQyVG9rZW5pemVyRmFzdApwcmludChmIkxvYWRpbmcge01PREVMX05BTUV9IC4uLiIpCm1vZGVsID0gQXV0b01vZGVsRm9yQ2F1c2FsTE0uZnJvbV9wcmV0cmFpbmVkKAogICAgTU9ERUxfTkFNRSwgdHJ1c3RfcmVtb3RlX2NvZGU9VFJVU1RfUkVNT1RFLCB0b3JjaF9kdHlwZT10aC5mbG9hdDMyKQp0b2tlbml6ZXIgPSBHUFQyVG9rZW5pemVyRmFzdC5mcm9tX3ByZXRyYWluZWQoTU9ERUxfTkFNRSwgdHJ1c3RfcmVtb3RlX2NvZGU9VFJVU1RfUkVNT1RFKQppZiB0b2tlbml6ZXIucGFkX3Rva2VuIGlzIE5vbmU6CiAgICB0b2tlbml6ZXIucGFkX3Rva2VuID0gdG9rZW5pemVyLmVvc190b2tlbgpQQURfSUQgPSB0b2tlbml6ZXIucGFkX3Rva2VuX2lkClZPQ0FCICA9IG1vZGVsLmNvbmZpZy52b2NhYl9zaXplCnByaW50KGYiVG9rZW5pemVyIHZvY2FiOiB7Vk9DQUJ9LCBwYWRfaWQ9e1BBRF9JRH0iKQoKbW9kZWwgPSBtb2RlbC50byhkZXZpY2UpCnRyeToKICAgIG1vZGVsLmNvbmZpZy51c2VfY2FjaGUgPSBGYWxzZQpleGNlcHQgRXhjZXB0aW9uOgogICAgcGFzcwp0cnk6CiAgICBtb2RlbC5ncmFkaWVudF9jaGVja3BvaW50aW5nX2VuYWJsZSgpCiAgICBwcmludCgiR3JhZGllbnQgY2hlY2twb2ludGluZyBlbmFibGVkIOKckyIpCmV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgIHByaW50KGYi4pqgIGdyYWRpZW50X2NoZWNrcG9pbnRpbmcgdW5hdmFpbGFibGUgKHtlfSkiKQoKaWYgdGguY3VkYS5pc19hdmFpbGFibGUoKSBhbmQgdGguY3VkYS5kZXZpY2VfY291bnQoKSA+IDE6CiAgICBwcmludChmIlVzaW5nIHt0aC5jdWRhLmRldmljZV9jb3VudCgpfSBHUFVzIHZpYSBEYXRhUGFyYWxsZWwiKQogICAgbW9kZWwgPSB0aC5ubi5EYXRhUGFyYWxsZWwobW9kZWwpCl9jb3JlID0gbW9kZWwubW9kdWxlIGlmIGlzaW5zdGFuY2UobW9kZWwsIHRoLm5uLkRhdGFQYXJhbGxlbCkgZWxzZSBtb2RlbApwcmludChmIlBhcmFtZXRlcnM6IHtzdW0ocC5udW1lbCgpIGZvciBwIGluIG1vZGVsLnBhcmFtZXRlcnMoKSkvMWU2Oi4xZn1NIikKCiMg4pSA4pSAIEJ1aWxkIGRhdGFzZXQg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSAClBST01QVCA9ICIjIyMgSW5zdHJ1Y3Rpb246XG57aW5zdHJ1Y3Rpb259XG5cbiMjIyBSZXNwb25zZTpcbntyZXNwb25zZX0iCnNhbXBsZXMgPSBbXQoKbG9jYWxfcGF0aCA9IChST09UIC8gZHNfY2ZnWyJsb2NhbF9wYXRoIl0pLnJlc29sdmUoKQppZiBsb2NhbF9wYXRoLmV4aXN0cygpOgogICAgd2l0aCBvcGVuKGxvY2FsX3BhdGgsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAgZm9yIGxpbmUgaW4gZjoKICAgICAgICAgICAgbGluZSA9IGxpbmUuc3RyaXAoKQogICAgICAgICAgICBpZiBub3QgbGluZToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIG9iaiA9IGpzb24ubG9hZHMobGluZSkKICAgICAgICAgICAgaW5zdCwgb3V0ID0gY2xlYW4ob2JqLmdldCgiaW5zdHJ1Y3Rpb24iLCAiIikpLCBjbGVhbihvYmouZ2V0KCJvdXRwdXQiLCAiIikpCiAgICAgICAgICAgIGlmIGluc3QgYW5kIG91dDoKICAgICAgICAgICAgICAgIHNhbXBsZXMuYXBwZW5kKChpbnN0LCBvdXQpKQogICAgcHJpbnQoZiJMb2FkZWQge2xlbihzYW1wbGVzKX0gbG9jYWwgUSZBIGZyb20ge2xvY2FsX3BhdGh9IikKZWxzZToKICAgIHByaW50KGYi4pqgIGxvY2FsIGRhdGEgbm90IGZvdW5kOiB7bG9jYWxfcGF0aH0iKQoKc2FtcGxlcyA9IHNhbXBsZXMgKiBpbnQoZHNfY2ZnLmdldCgibG9jYWxfcmVwZWF0IiwgMSkpCgpjYW5kaWRhdGVzID0gW10KaWYgZHNfY2ZnLmdldCgiaGZfZGF0YXNldCIpOgogICAgY2FuZGlkYXRlcy5hcHBlbmQoZHNfY2ZnWyJoZl9kYXRhc2V0Il0pCmNhbmRpZGF0ZXMgKz0gWyJhcmJtbC9BTFBBQ0FfQVIiLCAiTTRhbGkvYXJhYmljLWluc3RydWN0IiwgIk9BTEwvQWxwYWNhLUFyYWJpYyIsICJ2aW5lZXRzaGFybWEvYXJhYmljLWluc3RydWN0Il0KaGZfbG9hZGVkID0gRmFsc2UKZm9yIGhmX25hbWUgaW4gY2FuZGlkYXRlczoKICAgIHRyeToKICAgICAgICBmcm9tIGRhdGFzZXRzIGltcG9ydCBsb2FkX2RhdGFzZXQKICAgICAgICBwcmludChmIkxvYWRpbmcgSEYgZGF0YXNldCB7aGZfbmFtZX0gLi4uIikKICAgICAgICBsb2FkX2t3YXJncyA9IHsicGF0aCI6IGhmX25hbWUsICJzcGxpdCI6IGRzX2NmZy5nZXQoImhmX3NwbGl0IiwgInRyYWluIil9CiAgICAgICAgaWYgSEZfVE9LRU46CiAgICAgICAgICAgIGxvYWRfa3dhcmdzWyJ0b2tlbiJdID0gSEZfVE9LRU4KICAgICAgICBoZl9kcyA9IGxvYWRfZGF0YXNldCgqKmxvYWRfa3dhcmdzKQogICAgICAgIGluc3RfZiwgb3V0X2YgPSBkc19jZmcuZ2V0KCJoZl9pbnN0cnVjdGlvbl9maWVsZCIsICJpbnN0cnVjdGlvbiIpLCBkc19jZmcuZ2V0KCJoZl9vdXRwdXRfZmllbGQiLCAib3V0cHV0IikKICAgICAgICBhZGRlZCA9IDAKICAgICAgICBmb3Igcm93IGluIGhmX2RzOgogICAgICAgICAgICBpbnN0LCBvdXQgPSBjbGVhbihyb3cuZ2V0KGluc3RfZiwgIiIpIG9yICIiKSwgY2xlYW4ocm93LmdldChvdXRfZiwgIiIpIG9yICIiKQogICAgICAgICAgICBpZiBpbnN0IGFuZCBvdXQgYW5kIGxlbihvdXQpIDwgNjAwOgogICAgICAgICAgICAgICAgc2FtcGxlcy5hcHBlbmQoKGluc3QsIG91dCkpOyBhZGRlZCArPSAxCiAgICAgICAgICAgIGlmIGRzX2NmZy5nZXQoImhmX21heCIpIGFuZCBhZGRlZCA+PSBkc19jZmdbImhmX21heCJdOgogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICBwcmludChmIkFkZGVkIHthZGRlZH0gc2FtcGxlcyBmcm9tIEhGIGRhdGFzZXQge2hmX25hbWV9IikKICAgICAgICBoZl9sb2FkZWQgPSBUcnVlCiAgICAgICAgYnJlYWsKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICBwcmludChmIuKaoCBkYXRhc2V0IHtoZl9uYW1lfSB1bmF2YWlsYWJsZSAoe2V9KTsgdHJ5aW5nIG5leHQiKQppZiBub3QgaGZfbG9hZGVkOgogICAgcHJpbnQoIuKaoCBObyBIRiBkYXRhc2V0IGxvYWRlZDsgdXNpbmcgbG9jYWwgZGF0YSBvbmx5IikKCmlmIG5vdCBzYW1wbGVzOgogICAgcmFpc2UgU3lzdGVtRXhpdCgiTm8gdHJhaW5pbmcgc2FtcGxlcyBmb3VuZCDigJQgYWJvcnRpbmcuIikKcmFuZG9tLnNodWZmbGUoc2FtcGxlcykKcHJpbnQoZiJUb3RhbCB0cmFpbmluZyBzYW1wbGVzOiB7bGVuKHNhbXBsZXMpOix9IikKCmNsYXNzIEluc3RydWN0RGF0YXNldChEYXRhc2V0KToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBzYW1wbGVzLCBzZXFfbGVuPU1BWF9MRU4pOgogICAgICAgIHNlbGYuc2VxX2xlbiA9IHNlcV9sZW4KICAgICAgICBwcmludChmIlRva2VuaXppbmcge2xlbihzYW1wbGVzKTosfSBzYW1wbGVzIC4uLiIpCiAgICAgICAgdG9rcyA9IFtdCiAgICAgICAgZm9yIGluc3QsIHJlc3AgaW4gc2FtcGxlczoKICAgICAgICAgICAgdGV4dCA9IFBST01QVC5mb3JtYXQoaW5zdHJ1Y3Rpb249aW5zdCwgcmVzcG9uc2U9cmVzcCkKICAgICAgICAgICAgdG9rcy5leHRlbmQodG9rZW5pemVyLmVuY29kZSh0ZXh0LCBhZGRfc3BlY2lhbF90b2tlbnM9RmFsc2UpKQogICAgICAgIHNlbGYuZGF0YSA9IHRoLnRlbnNvcih0b2tzLCBkdHlwZT10aC5sb25nKQogICAgICAgIHNlbGYubiA9IChsZW4oc2VsZi5kYXRhKSAtIDEpIC8vIHNlcV9sZW4KICAgICAgICBpZiBzZWxmLm4gPT0gMDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIk5vdCBlbm91Z2ggdG9rZW5zOiB7bGVuKHNlbGYuZGF0YSl9IDwge3NlcV9sZW4rMX0iKQogICAgICAgIHByaW50KGYiVG9rZW5zOiB7bGVuKHNlbGYuZGF0YSk6LH0g4oaSIHtzZWxmLm46LH0gY2h1bmtzIikKCiAgICBkZWYgX19sZW5fXyhzZWxmKToKICAgICAgICByZXR1cm4gc2VsZi5uCgogICAgZGVmIF9fZ2V0aXRlbV9fKHNlbGYsIGkpOgogICAgICAgIHMgPSBpICogc2VsZi5zZXFfbGVuCiAgICAgICAgY2h1bmsgPSBzZWxmLmRhdGFbczpzICsgc2VsZi5zZXFfbGVuICsgMV0KICAgICAgICBpZiBsZW4oY2h1bmspIDwgc2VsZi5zZXFfbGVuICsgMToKICAgICAgICAgICAgY2h1bmsgPSB0aC5jYXQoW2NodW5rLCB0aC5mdWxsKChzZWxmLnNlcV9sZW4gKyAxIC0gbGVuKGNodW5rKSwpLCBQQURfSUQsIGR0eXBlPXRoLmxvbmcpXSkKICAgICAgICByZXR1cm4gY2h1bmsKCnNwbGl0ICAgICA9IGludChsZW4oc2FtcGxlcykgKiBkc19jZmcuZ2V0KCJ0cmFpbl9zcGxpdCIsIDAuOTUpKQp0cmFpbl9kcyAgPSBJbnN0cnVjdERhdGFzZXQoc2FtcGxlc1s6c3BsaXRdKQpldmFsX2RzICAgPSBJbnN0cnVjdERhdGFzZXQoc2FtcGxlc1tzcGxpdDpdKQpiYXRjaF9zaXplID0gdHJhaW5fY2ZnWyJiYXRjaF9zaXplIl0KdHJhaW5fbG9hZGVyID0gRGF0YUxvYWRlcih0cmFpbl9kcywgYmF0Y2hfc2l6ZT1iYXRjaF9zaXplLCBzaHVmZmxlPVRydWUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgbnVtX3dvcmtlcnM9MiwgcGluX21lbW9yeT10aC5jdWRhLmlzX2F2YWlsYWJsZSgpLCBkcm9wX2xhc3Q9VHJ1ZSkKZXZhbF9sb2FkZXIgID0gRGF0YUxvYWRlcihldmFsX2RzLCAgYmF0Y2hfc2l6ZT1iYXRjaF9zaXplLCBzaHVmZmxlPUZhbHNlLAogICAgICAgICAgICAgICAgICAgICAgICAgIG51bV93b3JrZXJzPTIsIHBpbl9tZW1vcnk9dGguY3VkYS5pc19hdmFpbGFibGUoKSwgZHJvcF9sYXN0PVRydWUpCnByaW50KCJEYXRhc2V0IHJlYWR5IOKckyIpCgojIOKUgOKUgCBUcmFpbmluZyBzZXR1cCDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKTUFYX0hPVVJTICA9IHRyYWluX2NmZ1sibWF4X2hvdXJzIl0KR1JBRF9BQ0NVTSA9IHRyYWluX2NmZ1siZ3JhZF9hY2N1bSJdCk1BWF9TVEVQUyAgPSB0cmFpbl9jZmdbIm1heF9zdGVwcyJdCldBUk1VUCAgICAgPSB0cmFpbl9jZmdbIndhcm11cF9zdGVwcyJdCkxPR19FVkVSWSAgPSB0cmFpbl9jZmdbImxvZ19ldmVyeSJdCkVWQUxfRVZFUlkgPSB0cmFpbl9jZmdbImV2YWxfZXZlcnkiXQpMUiAgICAgICAgID0gdHJhaW5fY2ZnWyJsciJdCkxSX01JTiAgICAgPSB0cmFpbl9jZmdbImxyX21pbiJdCk9WRVJGSVRfUEFUPSB0cmFpbl9jZmcuZ2V0KCJvdmVyZml0X3BhdGllbmNlIiwgNSkKCmRlY2F5LCBub19kZWNheSA9IFtdLCBbXQpmb3IgbmFtZSwgcCBpbiBtb2RlbC5uYW1lZF9wYXJhbWV0ZXJzKCk6CiAgICBpZiBub3QgcC5yZXF1aXJlc19ncmFkOgogICAgICAgIGNvbnRpbnVlCiAgICBpZiBwLmRpbSgpIDwgMiBvciBhbnkoeCBpbiBuYW1lIGZvciB4IGluIFsibG4iLCAiYmlhcyIsICJlbWJlZCIsICJMYXllck5vcm0iXSk6CiAgICAgICAgbm9fZGVjYXkuYXBwZW5kKHApCiAgICBlbHNlOgogICAgICAgIGRlY2F5LmFwcGVuZChwKQoKZGVmIF9tYWtlX29wdGltaXplcihkZWNheSwgbm9fZGVjYXkpOgogICAgdHJ5OgogICAgICAgIGZyb20gYml0c2FuZGJ5dGVzLm9wdGltIGltcG9ydCBBZGFtVzhiaXQKICAgICAgICByZXR1cm4gQWRhbVc4Yml0KFsKICAgICAgICAgICAgeyJwYXJhbXMiOiBkZWNheSwgIndlaWdodF9kZWNheSI6IDAuMDF9LAogICAgICAgICAgICB7InBhcmFtcyI6IG5vX2RlY2F5LCAid2VpZ2h0X2RlY2F5IjogMC4wfSwKICAgICAgICBdLCBscj1MUiwgYmV0YXM9KDAuOSwgMC45NSkpLCBUcnVlCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHBhc3MKICAgIHRyeToKICAgICAgICBzdWJwcm9jZXNzLmNoZWNrX2NhbGwoW3N5cy5leGVjdXRhYmxlLCAiLW0iLCAicGlwIiwgImluc3RhbGwiLCAiLXEiLCAiYml0c2FuZGJ5dGVzIl0pCiAgICAgICAgZnJvbSBiaXRzYW5kYnl0ZXMub3B0aW0gaW1wb3J0IEFkYW1XOGJpdAogICAgICAgIHJldHVybiBBZGFtVzhiaXQoWwogICAgICAgICAgICB7InBhcmFtcyI6IGRlY2F5LCAid2VpZ2h0X2RlY2F5IjogMC4wMX0sCiAgICAgICAgICAgIHsicGFyYW1zIjogbm9fZGVjYXksICJ3ZWlnaHRfZGVjYXkiOiAwLjB9LAogICAgICAgIF0sIGxyPUxSLCBiZXRhcz0oMC45LCAwLjk1KSksIFRydWUKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICBwcmludChmIuKaoCBiaXRzYW5kYnl0ZXMgdW5hdmFpbGFibGUgKHtlfSk7IHVzaW5nIHN0YW5kYXJkIEFkYW1XIikKICAgICAgICByZXR1cm4gdGgub3B0aW0uQWRhbVcoWwogICAgICAgICAgICB7InBhcmFtcyI6IGRlY2F5LCAid2VpZ2h0X2RlY2F5IjogMC4wMX0sCiAgICAgICAgICAgIHsicGFyYW1zIjogbm9fZGVjYXksICJ3ZWlnaHRfZGVjYXkiOiAwLjB9LAogICAgICAgIF0sIGxyPUxSLCBiZXRhcz0oMC45LCAwLjk1KSwgZnVzZWQ9dGguY3VkYS5pc19hdmFpbGFibGUoKSksIEZhbHNlCgpvcHRpbWl6ZXIsIF91c2VfOGJpdCA9IF9tYWtlX29wdGltaXplcihkZWNheSwgbm9fZGVjYXkpCnByaW50KCJVc2luZyA4LWJpdCBBZGFtVyAoYml0c2FuZGJ5dGVzKSDinJMiIGlmIF91c2VfOGJpdCBlbHNlICJVc2luZyBzdGFuZGFyZCBBZGFtVyIpCnNjYWxlciA9IHRoLmFtcC5HcmFkU2NhbGVyKCJjdWRhIiwgZW5hYmxlZD10aC5jdWRhLmlzX2F2YWlsYWJsZSgpKQoKZGVmIGdldF9scihzdGVwKToKICAgIGlmIHN0ZXAgPCBXQVJNVVA6CiAgICAgICAgcmV0dXJuIExSICogKHN0ZXAgKyAxKSAvIFdBUk1VUAogICAgcHJvZ3Jlc3MgPSAoc3RlcCAtIFdBUk1VUCkgLyBtYXgoMSwgTUFYX1NURVBTIC0gV0FSTVVQKQogICAgY29zaW5lID0gMC41ICogKDEgKyBtYXRoLmNvcyhtYXRoLnBpICogbWluKHByb2dyZXNzLCAxLjApKSkKICAgIHJldHVybiBMUl9NSU4gKyAoTFIgLSBMUl9NSU4pICogY29zaW5lCgpAdGgubm9fZ3JhZCgpCmRlZiBldmFsdWF0ZShuPTMwKToKICAgIG1vZGVsLmV2YWwoKTsgdG90YWwgPSAwLjA7IG5iID0gMAogICAgdHJ5OgogICAgICAgIGZvciBpLCBiYXRjaCBpbiBlbnVtZXJhdGUoZXZhbF9sb2FkZXIpOgogICAgICAgICAgICBpZiBpID49IG46CiAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICB4ID0gYmF0Y2hbOiwgOi0xXS50byhkZXZpY2UpOyB5ID0gYmF0Y2hbOiwgMTpdLnRvKGRldmljZSkKICAgICAgICAgICAgd2l0aCB0aC5hbXAuYXV0b2Nhc3QoImN1ZGEiLCBlbmFibGVkPXRoLmN1ZGEuaXNfYXZhaWxhYmxlKCkpOgogICAgICAgICAgICAgICAgbG9zcyA9IEYuY3Jvc3NfZW50cm9weShtb2RlbCh4KS5sb2dpdHMucmVzaGFwZSgtMSwgVk9DQUIpLCB5LnJlc2hhcGUoLTEpLCBpZ25vcmVfaW5kZXg9UEFEX0lEKQogICAgICAgICAgICB0b3RhbCArPSBsb3NzLml0ZW0oKTsgbmIgKz0gMQogICAgZmluYWxseToKICAgICAgICBtb2RlbC50cmFpbigpCiAgICByZXR1cm4gdG90YWwgLyBtYXgobmIsIDEpCgpkZWYgX2NvcHlfY3VzdG9tX2NvZGUoZHN0KToKICAgIGNvcGllZCA9IFtdCiAgICB0cnk6CiAgICAgICAgbW9kID0gc3lzLm1vZHVsZXNbbW9kZWwuX19jbGFzc19fLl9fbW9kdWxlX19dCiAgICAgICAgZiA9IGluc3BlY3QuZ2V0ZmlsZShtb2QpCiAgICAgICAgc2h1dGlsLmNvcHkoZiwgb3MucGF0aC5qb2luKGRzdCwgb3MucGF0aC5iYXNlbmFtZShmKSkpCiAgICAgICAgY29waWVkLmFwcGVuZChvcy5wYXRoLmJhc2VuYW1lKGYpKQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgIHByaW50KGYi4pqgIGNvcHkgbW9kZWwgY29kZSBmYWlsZWQ6IHtlfSIpCiAgICB0cnk6CiAgICAgICAgY2NscyA9IF9jb3JlLmNvbmZpZy5fX2NsYXNzX18KICAgICAgICBjbW9kID0gc3lzLm1vZHVsZXNbY2Nscy5fX21vZHVsZV9fXQogICAgICAgIGYgPSBpbnNwZWN0LmdldGZpbGUoY21vZCkKICAgICAgICBzaHV0aWwuY29weShmLCBvcy5wYXRoLmpvaW4oZHN0LCBvcy5wYXRoLmJhc2VuYW1lKGYpKSkKICAgICAgICBjb3BpZWQuYXBwZW5kKG9zLnBhdGguYmFzZW5hbWUoZikpCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgcHJpbnQoZiLimqAgY29weSBjb25maWcgY29kZSBmYWlsZWQ6IHtlfSIpCiAgICBpZiBjb3BpZWQ6CiAgICAgICAgcHJpbnQoZiIgIOKckyBjb3BpZWQgY3VzdG9tIGNvZGU6IHtjb3BpZWR9IikKCmRlZiBzYXZlX2Jlc3Qoc3RlcCwgbG9zcyk6CiAgICBfY29yZS5zYXZlX3ByZXRyYWluZWQoU0FWRV9ESVIpCiAgICB0b2tlbml6ZXIuc2F2ZV9wcmV0cmFpbmVkKFNBVkVfRElSKQogICAgX2NvcHlfY3VzdG9tX2NvZGUoc3RyKFNBVkVfRElSKSkKICAgIHRoLnNhdmUoeyJzdGVwIjogc3RlcCwgImxvc3MiOiBsb3NzfSwgU0FWRV9ESVIgLyAidHJhaW5fc3RhdGUucHQiKQogICAgcHJpbnQoZiIgIOKckyBzYXZlZCBmaW5ldHVuZWQgbW9kZWwgLT4ge1NBVkVfRElSfSAoc3RlcD17c3RlcDosfSkiKQoKc3RlcCA9IDA7IGJlc3RfZXZhbCA9IGZsb2F0KCJpbmYiKTsgZXZhbF9oaXN0b3J5ID0gW107IGRvbmUgPSBGYWxzZQp0MCA9IHRpbWUudGltZSgpOyBtb2RlbC50cmFpbigpCmxvZ19wYXRoID0gU0FWRV9ESVIgLyAiZmluZXR1bmVfbG9nLmpzb25sIgpwcmludChmIlxueyc9Jyo1MH1cbkZpbmV0dW5lIHtNT0RFTF9OQU1FfSB8IHtNQVhfSE9VUlN9aCB8IGxyPXtMUn1cbnsnPScqNTB9XG4iKQpsb2FkZXJfaXRlciA9IGl0ZXIodHJhaW5fbG9hZGVyKQoKd2hpbGUgc3RlcCA8IE1BWF9TVEVQUyBhbmQgbm90IGRvbmU6CiAgICBsciA9IGdldF9scihzdGVwKQogICAgZm9yIGcgaW4gb3B0aW1pemVyLnBhcmFtX2dyb3VwczoKICAgICAgICBnWyJsciJdID0gbHIKICAgIG9wdGltaXplci56ZXJvX2dyYWQoc2V0X3RvX25vbmU9VHJ1ZSkKICAgIGxvc3NfdmFsID0gMC4wCiAgICBmb3IgXyBpbiByYW5nZShHUkFEX0FDQ1VNKToKICAgICAgICB0cnk6CiAgICAgICAgICAgIG1pY3JvID0gbmV4dChsb2FkZXJfaXRlcikKICAgICAgICBleGNlcHQgU3RvcEl0ZXJhdGlvbjoKICAgICAgICAgICAgbG9hZGVyX2l0ZXIgPSBpdGVyKHRyYWluX2xvYWRlcikKICAgICAgICAgICAgbWljcm8gPSBuZXh0KGxvYWRlcl9pdGVyKQogICAgICAgIHggPSBtaWNyb1s6LCA6LTFdLnRvKGRldmljZSwgbm9uX2Jsb2NraW5nPVRydWUpCiAgICAgICAgeSA9IG1pY3JvWzosIDE6XS50byhkZXZpY2UsIG5vbl9ibG9ja2luZz1UcnVlKQogICAgICAgIHdpdGggdGguYW1wLmF1dG9jYXN0KCJjdWRhIiwgZW5hYmxlZD10aC5jdWRhLmlzX2F2YWlsYWJsZSgpKToKICAgICAgICAgICAgbG9zcyA9IEYuY3Jvc3NfZW50cm9weShtb2RlbCh4KS5sb2dpdHMucmVzaGFwZSgtMSwgVk9DQUIpLCB5LnJlc2hhcGUoLTEpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGlnbm9yZV9pbmRleD1QQURfSUQpIC8gR1JBRF9BQ0NVTQogICAgICAgIHNjYWxlci5zY2FsZShsb3NzKS5iYWNrd2FyZCgpCiAgICAgICAgbG9zc192YWwgKz0gbG9zcy5pdGVtKCkKICAgIHNjYWxlci51bnNjYWxlXyhvcHRpbWl6ZXIpCiAgICBnbm9ybSA9IHRoLm5uLnV0aWxzLmNsaXBfZ3JhZF9ub3JtXyhtb2RlbC5wYXJhbWV0ZXJzKCksIDEuMCkKICAgIHNjYWxlci5zdGVwKG9wdGltaXplcik7IHNjYWxlci51cGRhdGUoKQogICAgc3RlcCArPSAxCgogICAgZWxhcHNlZCA9ICh0aW1lLnRpbWUoKSAtIHQwKSAvIDM2MDAKICAgIGlmIGVsYXBzZWQgPj0gTUFYX0hPVVJTOgogICAgICAgIF9jb3JlLnNhdmVfcHJldHJhaW5lZChTQVZFX0RJUik7IHRva2VuaXplci5zYXZlX3ByZXRyYWluZWQoU0FWRV9ESVIpCiAgICAgICAgX2NvcHlfY3VzdG9tX2NvZGUoc3RyKFNBVkVfRElSKSkKICAgICAgICB0aC5zYXZlKHsic3RlcCI6IHN0ZXAsICJsb3NzIjogbG9zc192YWx9LCBTQVZFX0RJUiAvICJ0cmFpbl9zdGF0ZS5wdCIpCiAgICAgICAgcHJpbnQoZiJcbuKPsSBUaW1lIGxpbWl0LiBTYXZlZCAtPiB7U0FWRV9ESVJ9IChzdGVwPXtzdGVwOix9KSIpCiAgICAgICAgZG9uZSA9IFRydWU7IGJyZWFrCgogICAgaWYgc3RlcCAlIExPR19FVkVSWSA9PSAwOgogICAgICAgIHBwbCA9IG1hdGguZXhwKG1pbihsb3NzX3ZhbCwgMjApKQogICAgICAgIHByaW50KGYic3RlcD17c3RlcDo1ZH0gfCBsb3NzPXtsb3NzX3ZhbDouNGZ9IHwgcHBsPXtwcGw6LjFmfSB8IGxyPXtscjouMmV9IHwgZ25vcm09e2dub3JtOi4yZn0gfCB7ZWxhcHNlZDouMmZ9aCIpCiAgICAgICAgd2l0aCBvcGVuKGxvZ19wYXRoLCAiYSIpIGFzIGY6CiAgICAgICAgICAgIGYud3JpdGUoanNvbi5kdW1wcyh7InN0ZXAiOiBzdGVwLCAibG9zcyI6IGxvc3NfdmFsLCAicHBsIjogcHBsLCAibHIiOiBsciwgImdub3JtIjogZmxvYXQoZ25vcm0pLCAiZWxhcHNlZF9oIjogZWxhcHNlZH0pICsgIlxuIikKCiAgICBpZiBzdGVwICUgRVZBTF9FVkVSWSA9PSAwOgogICAgICAgIGV2ID0gZXZhbHVhdGUoKTsgcHBsID0gbWF0aC5leHAobWluKGV2LCAyMCkpCiAgICAgICAgcHJpbnQoZiIgIOKGsyBldmFsPXtldjouNGZ9IHwgcHBsPXtwcGw6LjFmfSIpCiAgICAgICAgd2l0aCBvcGVuKGxvZ19wYXRoLCAiYSIpIGFzIGY6CiAgICAgICAgICAgIGYud3JpdGUoanNvbi5kdW1wcyh7InN0ZXAiOiBzdGVwLCAiZXZhbF9sb3NzIjogZXZ9KSArICJcbiIpCiAgICAgICAgaWYgZXYgPCBiZXN0X2V2YWw6CiAgICAgICAgICAgIGJlc3RfZXZhbCA9IGV2OyBzYXZlX2Jlc3Qoc3RlcCwgZXYpCiAgICAgICAgZXZhbF9oaXN0b3J5LmFwcGVuZChldikKICAgICAgICBpZiBsZW4oZXZhbF9oaXN0b3J5KSA+IE9WRVJGSVRfUEFUIGFuZCBhbGwoZXZhbF9oaXN0b3J5Wy1pXSA+IGV2YWxfaGlzdG9yeVstaS0xXSBmb3IgaSBpbiByYW5nZSgxLCBPVkVSRklUX1BBVCsxKSk6CiAgICAgICAgICAgIHByaW50KGYiXG7imqAgT1ZFUkZJVFRJTkcge09WRVJGSVRfUEFUfXgg4oCUIHN0b3BwaW5nLiBCZXN0IGV2YWw9e2Jlc3RfZXZhbDouNGZ9IikKICAgICAgICAgICAgZG9uZSA9IFRydWU7IGJyZWFrCgppZiBub3QgZG9uZToKICAgIHByaW50KGYiXG5Eb25lLiBzdGVwPXtzdGVwOix9IHwgYmVzdF9ldmFsPXtiZXN0X2V2YWw6LjRmfSIpCg==').decode()
open('/kaggle/working/pg/training/finetune_hf.py','w',encoding='utf-8').write(code)
print('  done. Script bytes:', len(code))


In [ ]:
!cd pg && python -u training/finetune_hf.py

## Done — download the model

Open the **Output** tab (right side) → `pg/weights_hf/` → download the folder. Then test locally:

`python scripts/test_hf.py --model_dir /path/to/weights_hf`